In [0]:
#reload silver tables fresh

orders_silver = spark.read.format("delta").load("/Volumes/workspace/default/raw_uploads/silver/orders")
customers_silver = spark.read.format("delta").load("/Volumes/workspace/default/raw_uploads/silver/customers")
items_silver = spark.read.format("delta").load("/Volumes/workspace/default/raw_uploads/silver/order_items")
payments_silver = spark.read.format("delta").load("/Volumes/workspace/default/raw_uploads/silver/order_payments")
holidays_bronze = spark.read.format("delta").load("/Volumes/workspace/default/raw_uploads/bronze/holidays")

from pyspark.sql import functions as F

In [0]:
joined_check = orders_silver.join(customers_silver, on="customer_id", how="inner")
joined_check.printSchema()

In [0]:
orders_silver.printSchema()

In [0]:
sales_by_region = (
    orders_silver
    .join(payments_silver, on="order_id", how="inner")
    .groupBy("customer_state")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("payment_value").alias("total_revenue"),
        F.round(F.avg("payment_value"), 2).alias("avg_order_value"),
    )
    .orderBy(F.desc("total_revenue"))
)

sales_by_region.show(10)

In [0]:
#gold table 2 - daily order trends

daily_trends = (
    orders_silver
    .join(payments_silver, on="order_id", how="inner")
    .groupBy("order_date")
    .agg(
        F.count("order_id").alias("daily_order_count"),
        F.round(F.avg("payment_value"), 2).alias("avg_order_value"),
    )
    .orderBy("order_date")
)

daily_trends.show(10)

#this only joins orders_silver and payments_silver - neither of which independently carries a duplicate column, unlike the customer_state situation. order_date already lives cleanly in orders_silver

In [0]:
holiday_dates = holidays_bronze.select(
    F.col("date").cast("date").alias("holiday_date")
).distinct()

sales_with_holiday_flag = (
    daily_trends
    .join(
        holiday_dates,
        daily_trends.order_date == holiday_dates.holiday_date,
        how="left"
    )
    .withColumn(
        "is_holiday", F.when(F.col("holiday_date").isNotNull(), True).otherwise(False)
    )
    .drop("holiday_date")
)

holiday_comparison = (
    sales_with_holiday_flag
    .groupBy("is_holiday")
    .agg(
        F.round(F.avg("daily_order_count"), 1).alias("avg_daily_orders"),
        F.round(F.avg("avg_order_value"), 2).alias("avg_order_value"),
    )
)
holiday_comparison.show()

#.cast("date") - converts the holiday APIs date column (likely stored as text) into a real date type, so it can be compared agaisnt order_date
#daily_trends.order_date == holiday_dates.holiday_date - written this way (not on = "..") bc the two columns ave diff names
#F.when(..).otherwise(..) - sparks if/else, applied per row

In [0]:
sales_by_region.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/gold/sales_by_region"
)
daily_trends.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/gold/daily_trends"
)
holiday_comparison.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/gold/holiday_comparison"
)

print("Gold tables written: sales_by_region, daily_trends, holiday_comparison")

In [0]:
sales_by_region.display()
holiday_comparison.display()

In [0]:
orders_silver.printSchema()

In [0]:
#A payment method breakdown table
payment_method_breakdown = (
    orders_silver
    .join(payments_silver, on="order_id", how="inner")
    .groupBy("payment_type")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("payment_value"), 2).alias("total_revenue"),
        F.round(F.avg("payment_value"), 2).alias("avg_order_value"),
    )
    .orderBy(F.desc("total_orders"))
)

payment_method_breakdown.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/raw_uploads/gold/payment_method_breakdown"
)
payment_method_breakdown.show()



In [0]:
print("=== GOLD LAYER SUMMARY ===")
print(f"Sales by region - states covered: {sales_by_region.count()}")
print(f"Daily trends - days covered: {daily_trends.count()}")
print(f"Payment methods - types covered: {payment_method_breakdown.count()}")
print("\nTop region by revenue:")
sales_by_region.show(1)
print("\nHoliday vs non-holiday comparison:")
holiday_comparison.show()

Here's the plain-English breakdown of everything from today's Gold layer work — same style as your Day 4 notes.

## What you built today
- **Sales by region** — total orders, total revenue, and average order value, broken down by customer state.
- **Daily trends** — how order volume and average order value change day by day across the whole dataset.
- **Holiday comparison** — whether Brazilian public holidays actually correlate with more or fewer orders (this is the table that finally *uses* your Week 1 holiday API data for something).
- **Payment method breakdown** — how customers pay, and whether payment method relates to order value.

## Concepts to remember

- **Business question first, code second.** Before writing a single line of Gold-layer code today, you wrote down what question each table was supposed to answer. This is the difference between "I ran some aggregations" and "I built reporting someone would actually use" — always worth doing this step before jumping into code, not after.

- **Joins can carry "hidden" columns you don't expect.** This was the big lesson today. `orders_silver` already had a `customer_state` column baked in from Day 3 (when you validated orders against customers). When you tried to join it with `customers_silver` again today, Spark got confused because now there were *two* columns with that same name — it didn't know which one you meant.
  - **The fix:** you either select only the specific columns you need before joining, or in this case, realize you didn't need the second join at all since the column was already there.
  - **The habit to take forward:** before joining two tables, check what columns each one *actually* has (`.printSchema()`) — don't assume a table only contains what its name suggests.

- **Different columns, different join syntax.** For most of your joins so far, you've used the simple `on="column_name"` shortcut — but that only works when both tables use the *exact same* column name. Today, joining `daily_trends` (with `order_date`) against `holiday_dates` (with `holiday_date`) needed the full explicit version instead: `daily_trends.order_date == holiday_dates.holiday_date`, since the names didn't match.

- **`.cast("date")`** — a new tool today. Converts a column from one data type to another — here, turning the holiday API's date (likely stored as plain text) into an actual date type, so it could be properly compared against your order dates.

- **`F.when(...).otherwise(...)`** — Spark's version of an if/else, applied to every row at once. You used this to create the `is_holiday` True/False flag: "if there's a matching holiday date, mark True, otherwise False."

- **`F.round(value, 2)`** — rounds numbers to a set number of decimal places. Used throughout today so your dollar amounts read like $87.34, not some ugly 12-digit decimal.

## Mistake/lesson to remember for next time

- **The exact same "Silver carries hidden columns forward" issue will likely show up again** as you keep building more Gold tables — worth checking `printSchema()` proactively on any table before joining it into something new, rather than waiting for the ambiguous-column error to tell you.


In [0]:
reference_date = orders_silver.agg(F.max("order_date")).collect()[0][0]
print(reference_date)

In [0]:
recency_df = (
    orders_silver.groupBy("customer_unique_id")
    .agg(F.max("order_date").alias("last_order_date"))
    .withColumn("recency_days", F.datediff(F.lit(reference_date), F.col("last_order_date")))
)

recency_df.show(5)

#recency = days since their last order. a smaller recency_days means they bought more recently - thats the "still engaged" signal. 
#group by customer_unique_id(not customer_id) matters here - list gives each order its own customer_id, but customer_unique_id is the stable id that identifies the same real person across multiple orders. if you grouped by customer_id instead, every customer would look like they only ever ordered once'


In [0]:
frequency_df = (
    orders_silver.groupBy("customer_unique_id")
    .agg(F.countDistinct("order_id").alias("frequency"))
)

frequency_df.show(5)

#frequency = how many orders they've made. a higher frequency means they're more engaged.

In [0]:
monetary_df = (
    orders_silver
    .join(payments_silver, on="order_id", how="inner")
    .groupBy("customer_unique_id")
    .agg(F.round(F.sum("payment_value"), 2).alias("monetary"))
)

monetary_df.show(5)

#payment_value lives in payment_silver, not orders_silver, so this needs a join first - same pattern you already used for payment_method_breeakdown. using total money spent (not just order count) means a customer with a few expensive orders correctly shows up as high-value, not just "frequent".

In [0]:
frequency_df.groupBy("frequency").count().orderBy("frequency").show()

In [0]:
rfm_df = (
    recency_df
    .join(frequency_df, on="customer_unique_id", how="inner")
    .join(monetary_df, on="customer_unique_id", how="inner")
)

rfm_df.show(5)

#why inner join here: every customer that shows up in oders_silver should have a recency a frequency, and a monetary value - theres no reason one would exist without the others, since they are all dirived from the same source table. if the row count don't match after this join, that's a signal something off upstream

In [0]:
print("RFM row count:", rfm_df.count())
print("Distinct customers in orders_silver:", orders_silver.select("customer_unique_id").distinct().count())

rfm_df.filter(
    (F.col("recency_days") < 0) |
    (F.col("monetary") <= 0) |
    (F.col("frequency") <= 0)
).show()

#the first two counts should match or be very close - if rfm_df has fewer rows, some customers got dropped during a join, which means data is missing. the filter checks for impossible values (neg recency, zero/neg spend) - catching that now is far cheaper than discovering it after tomorrow's clustering produces nonsense segments

In [0]:
rfm_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/raw_uploads/gold/rfm_features")

print("RFM table written successfully")

#why: persisting this means tomorrows clustering notebook can read rfm_features directly, without rerunning todays joins from scratch - keeps ML work cleanly separated from gold layer - prep 